In this assignment, you will evaluate both the outputs and the internal behavior of pre-trained Transformer architectures.

Your objectives are to:

*   Inspect Tokenization differences (BERT vs T5)(subword splits, sequence length, and special tokens).

*   Observe Contextual Behavior using BERT masked token probing (how predictions chnage with context) and T5 input formatting (e.g. the summarize:prefix)

*  Run abstractive summarization inference using a pre-trained Transformer summarization model on the provided dataset

*  Inspect self-attention and multi-head attention behavior by extracting and visualizing BERT attention weights, then interpreting differences across heads and contexts.

Submission Requirement: Submit (1) the executed notebook and (2) a short written interpretation section in markdown cells.

In [ ]:
import os, json
from datasets import load_dataset

def load_week6_test_data():
    ds = load_dataset("cnn_dailymail", "3.0.0", split="test[:200]")  # small subset for runtime
    data = [{"id": i, "article": ds[i]["article"], "highlights": ds[i]["highlights"]} for i in range(len(ds))]
    return data, "Loaded fallback dataset: cnn_dailymail test[:200]"

test_data, DATA_MSG = load_week6_test_data()
print(DATA_MSG)
print("Sample record keys:", test_data[0].keys())
print("Total records loaded:", len(test_data))

## Task 1: Dataset Reuse & Tokenization (15 points)

*  Correct loading of BERT/T5 tokenizers and generation of the 3-sample comparison table: 5 pts

*  Successful implementation of the summarize: prefix for T5 input processing: 5 pts

*   Clear explanation of subword markers (## vs _) and how they solve the OOV problem: 5 pts

In [ ]:

# ==================== Task 1: Dual-Model Tokenization ====================


from transformers import AutoTokenizer
import pandas as pd

# Category A Blanks: Select the pre-trained checkpoints
# Recommended: "bert-base-uncased" and "t5-small"
bert_name = "_____"
t5_name   = "_____"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)
t5_tokenizer   = AutoTokenizer.from_pretrained(t5_name)

# Category A Blank: Choose a standardized sequence length (e.g., 128)
MAX_LEN = _____

print(f"Tokenizers loaded: BERT ({bert_name}), T5 ({t5_name})")

In [ ]:
########## Tokenization Comparison Logic ##########

# Student Code Required
## Instructions: "T5 is a 'Text-to-Text' model. Unlike BERT, it requires a
# specific instruction prefix to know that it is performing a summarization task.
# Fill in the blanks to correctly apply this prefix and observe the subword markers.

TOKEN_PREFIX = "summarize: "

def tokenize_one(article_text: str):
    """
    Processes one article through both BERT and T5 tokenizers.
    """
    # BERT: Standard processing
    bert = bert_tokenizer(article_text, truncation=True, max_length=MAX_LEN)

    # T5: Apply the task-specific prefix (e.g., "summarize: ")
    t5 = t5_tokenizer(TOKEN_PREFIX + article_text, truncation=True, max_length=MAX_LEN)

    bert_tokens = bert_tokenizer.convert_ids_to_tokens(bert["input_ids"][:40])
    t5_tokens   = t5_tokenizer.convert_ids_to_tokens(t5["input_ids"][:40])

    return {
        "bert_len": len(bert["input_ids"]),
        "t5_len": len(t5["input_ids"]),
        "bert_tokens_preview": " ".join(bert_tokens),
        "t5_tokens_preview": " ".join(t5_tokens),
    }

# Execution: Select 3 samples from the test_data (Week 5 subset)
sample_articles = [test_data[i]["article"] for i in range(3)]
rows = [tokenize_one(a) for a in sample_articles]
df_tok = pd.DataFrame(rows)

# Display the comparison table
df_tok

Exercise:

After inspecting the dataframe above, provide a short professional response to the following:

*  Subword Markers: Identify the specific markers BERT and T5 use for subwords. Why are these different?

*  The OOV Solution: Explain how subword tokenization prevents the 'Out-of-Vocabulary' (OOV) error. For example, if the word 'supercalifragilistic' was never seen during training, how would BERT handle it compared to the Keras tokenizer from Week 4?

*   Context & Attention: Why must the text be tokenized in this way before the Self-Attention mechanism can calculate relationships between words?

## Task 2: Contextual Behavior Probing (30 Points)

In this task, you will explore the revolutionary shift from Static to Dynamic Contextual Embeddings. You will use a pre-trained BERT model to 'probe' how the meaning of a word shifts based on its neighbors.

*   Successful implementation of the fill-mask pipeline and execution on both test sentences: 10 pts

*   AInterpretation (Sensitivity): Accurate description of how predictions shift between financial and geographical contexts: 10 pts

*  Interpretation (Mechanism): High-level explanation of how Query (Q) and Key (K) interaction facilitates this shift: 10 pts

In [ ]:
# Student code required
## Compare the top-5 predictions across contexts and explain why the
# distributions differ using the concept of contextual
# embeddings/self-attention


# ==================== Task 2: Contextual Behavior Probing ====================

from transformers import pipeline

# Initialize the fill-mask pipeline
fill_mask = pipeline("fill-mask", model="bert-base-uncased", top_k=5)

# Test contexts for the word "bank"
sentences = [
    "The [MASK] will not approve the loan because the credit score is too low.",
    "He sat on the river [MASK] and watched the water flow."
]

# Execution: Identify where to place the [MASK] to probe the word 'bank'
for s in sentences:
    preds = fill_mask(s)
    print(f"\nSentence: {s}")
    print("Top predictions:", [(p["token_str"], round(p["score"], 4)) for p in preds])

Exercise:

Based on your code output and the Week 6 teaching materials, answer the following:

*   Contextual Sensitivity: Describe how the top predictions changed between the 'loan' context and the 'river' context. Did BERT correctly identify the different meanings?

*   The Limitation of Static Embeddings: If we used Word2Vec or GloVe (Week 3) for this task, would the vector for 'bank' be different in these two sentences? Why is this a problem for summarization?

*   Self-Attention Mechanism: Briefly explain how the Query (Q) and Key (K) vectors interact (via dot product) to help BERT 'attend' to words like 'river' or 'loan' when updating the representation of the masked token

## Task 3: Pre-trained Transformer Summarization Inference (35 Points)

In this task, you will evaluate a state-of-the-art Transformer model to see the power of Transfer Learning. Unlike the 'from-scratch' models of Week 5, these models have been pre-trained on massive corpora and understand complex linguistic nuances.

*  Execution: Correct pipeline initialization and processing of N_EVAL articles (standardized to 50): 15 pts

*  Identification of "Abstractive" behavior (model-generated words not in source article): 10 pts

*  Explanation of why deterministic decoding (e.g., Beam Search) is preferred for news integrity: 10 pts

In [ ]:
# Student code required
# Instruction:Initialize the summarization pipeline.
# You must choose a recommended model checkpoint (Category A blank) and set
# the evaluation size (N\_EVAL$) to 50


# ==================== Task 3: Transformer Summarization ====================
import torch
from transformers import pipeline

# Choose a pre-trained model checkpoint
# Recommended: "sshleifer/distilbart-cnn-12-6" for speed or "BERT/T5" slower
model_checkpoint = "_____"


# Initialize the pipeline
# device=0 ensures GPU usage if available
summarizer = pipeline(
    "summarization",
    model=model_checkpoint,
    tokenizer=model_checkpoint,
    device=0 if torch.cuda.is_available() else -1
)


N_EVAL = 50

# Prepare test data subsets from the Week 5 dataset
articles = [test_data[i]["article"] for i in range(N_EVAL)]
refs     = [test_data[i]["highlights"] for i in range(N_EVAL)]
week6_ids = [test_data[i]["id"] for i in range(N_EVAL)] # Required for ID-based merge

# Execute Inference
preds = []

for i, a in enumerate(articles):
    # deterministic decoding via do_sample=False
    out = summarizer(a, max_length=80, min_length=20, do_sample=False)[0]["summary_text"]
    preds.append(out)

# Display a preview of generated summaries
print("Generated summaries:", len(preds))
print("\nSample prediction:\n", preds[0])

Exercise:

After inspecting your generated summaries, provide analysis of the following:

*   Abstractive vs. Extractive: Identify one instance where the model used a word or phrase that was not in the original article. How does this differ from the 'Lead-3' approach?

*  Sequential Bottleneck: Based on the Week 6 teaching slides, how does the Transformer's parallel processing (Self-Attention) help it summarize long articles better than the sequential LSTMs you used in Week 4?

*   Decoding Logic: We set do_sample=False. Explain why Deterministic Decoding (like Greedy Search or Beam Search) is often preferred for factual tasks like news summarization

In [ ]:
# We will verify WHY deterministic decoding is preferred by breaking the
# model with "Temperature".

print("\n--- Starting Controlled Chaos Experiment (N=3 Samples) ---\n")

# Use a small subset for visual inspection
chaos_articles = articles[:3]

# 1. Deterministic (Baseline)
print("1. Deterministic (Beam Search, num_beams=4):")
print(summarizer(chaos_articles[0], max_length=80, min_length=20,
                 do_sample=False, num_beams=4)[0]['summary_text'])

# 2. Low Temperature (Conservative Sampling)
print("\n2. Sampling (Temp=0.7 - Conservative):")
print(summarizer(chaos_articles[0], max_length=80, min_length=20,
                 do_sample=True, temperature=0.7)[0]['summary_text'])

# 3. High Temperature (Creative/Chaotic Sampling)
print("\n3. Sampling (Temp=1.5 - Chaotic):")
print(summarizer(chaos_articles[0], max_length=80, min_length=20,
                 do_sample=True, temperature=1.5)[0]['summary_text'])

### Analysis

**Compare the three outputs above.**

In Creative Writing (e.g., writing a novel), a High Temperature is desirable because it prevents the model from being boring. In Summarization, however, "creativity" often means "lying."

**Critical Analysis Questions:**
1.  **Hallucination check** : In the Temp=1.5 summary, did the model invent facts, entities (names/places), numbers, or claims that are not supported by the source article? If yes, quote 1–2 short phrases from the summary and point to the corresponding place in the article that shows the information is unsupported or missing.
2.  **Reliability & deployment** : Explain why a news organization would typically forbid do_sample=True in an automated summarization pipeline (even if the Temp=0.7 output reads slightly more natural). Your answer should reference risk, repeatability, and accountability.
3.  **Failure-mode analysis** :Identify two specific failure modes that increase under sampling (e.g., factual drift, repetition, missing key entities, incoherence). For each failure mode, provide one concrete example from your outputs (quote 1–2 short phrases) and clearly state which decoding setting produced it (baseline, Temp=0.7, or Temp=1.5)

## Task 4: Inspect Self-Attention & Multi-Head Attention (20 pts)

In the previous tasks, you treated the Transformer as a "black box" summarizer. In this final task, you will peer inside the "black box" to visualize the Self-Attention Mechanism we discussed in the lecture

Objective:
You will use the BERT model (an Encoder-only Transformer) to visualize how attention weights change based on context. This directly connects to the "River Bank" vs. "Financial Bank" example from the slides.

Visualize: Generate attention heatmaps for the word "bank" in two different sentences.

*   Analyze Heads: Calculate the "Entropy" of different attention heads to see how some focus narrowly (low entropy) while others attend broadly (high entropy): 10 pts

*   Interpret: Explain how these patterns prove that embeddings are dynamic, not static: 10 pts


In [ ]:

# ======= Task 4: Self-Attention & Multi-Head Attention Inspection (BERT) ======

import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

# 1. Setup Model (We use BERT-base for clear attention visualization)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True).to(device)
model.eval()

def get_attentions(text: str):
    """
    Returns:
      tokens: token strings
      attentions: tuple of length num_layers
    """
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())
    return tokens, outputs.attentions

def plot_attention_heatmap(attn_matrix, tokens, title="Attention Heatmap"):
    """
    Plots a heatmap of attention weights.
    """
    plt.figure(figsize=(10, 8))
    # Use 'Reds' or 'Viridis' to clearly show intensity
    plt.imshow(attn_matrix, aspect="auto", cmap='Reds')
    plt.xticks(range(len(tokens)), tokens, rotation=90, fontsize=12)
    plt.yticks(range(len(tokens)), tokens, fontsize=12)
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.show()

# ---- Part A: The "Bank" Experiment (Contextual Embeddings) ----
# We use the classic ambiguity example from the Lecture Slides
text_1 = "I went to the bank to deposit my paycheck."
text_2 = "I sat on the bank of the river and watched the water."

tokens1, attn1 = get_attentions(text_1)
tokens2, attn2 = get_attentions(text_2)

print(f"Sentence 1 Tokens: {tokens1}")
print(f"Sentence 2 Tokens: {tokens2}")

# ---- Part B: Visualize a Specific Head ----
# Try changing LAYER_IDX (0-11) and HEAD_IDX (0-11) to find interesting patterns
LAYER_IDX = 9    # Layers 8-10 often capture distinct syntactic relationships
HEAD_IDX = 5     # Arbitrary choice; feel free to explore

# Extract attention matrix for the chosen layer/head
# Shape: (batch, heads, seq, seq) -> We take [0, HEAD_IDX, :, :]
A1 = attn1[LAYER_IDX][0, HEAD_IDX].detach().cpu().numpy()
A2 = attn2[LAYER_IDX][0, HEAD_IDX].detach().cpu().numpy()

plot_attention_heatmap(A1, tokens1, title=f"Context 1 (Financial): Layer {LAYER_IDX} Head {HEAD_IDX}")
plot_attention_heatmap(A2, tokens2, title=f"Context 2 (Nature): Layer {LAYER_IDX} Head {HEAD_IDX}")

# ---- Part C: Multi-Head Entropy Analysis ----
# Entropy measures how "focused" or "spread out" the attention is.
# Low Entropy = Focused (Sharp attention on specific words)
# High Entropy = Diffuse (Broad attention over the whole sentence)

def attention_entropy(attn_head):
    # Clip values to avoid log(0)
    p = np.clip(attn_head, 1e-12, 1.0)
    # Entropy formula: -sum(p * log(p))
    return -np.sum(p * np.log(p), axis=-1).mean()

print(f"\n--- Multi-Head Entropy Analysis (Layer {LAYER_IDX}) ---")
head_entropies = []
for h in range(12):  # BERT-base has 12 heads
    # Get attention matrix for head 'h'
    Ah = attn1[LAYER_IDX][0, h].detach().cpu().numpy()
    entropy = attention_entropy(Ah)
    head_entropies.append((h, entropy))

# Sort by entropy to find the most "focused" heads
head_entropies.sort(key=lambda x: x[1])

print("Heads sorted by 'Focus' (Lowest Entropy to Highest):")
for h, e in head_entropies:
    print(f"Head {h:02d}: Entropy = {e:.4f}")

Exercise: Interpretation of Attention Mechanisms

Based on the heatmaps and entropy analysis you generated above, answer the following questions:

Visual Evidence (The "Bank" Ambiguity): Compare the heatmaps for Text 1 ("deposit my paycheck") and Text 2 ("river... watched the water").

Look specifically at the column for the word "bank".

Does the attention mechanism focus on different words in the two sentences? (e.g., does "bank" attend to "money/deposit" in the first, but "river/water" in the second?)

How does this visually demonstrate the concept of Contextual Embeddings (Slide 6)?

Multi-Head Specialization: Look at your Entropy scores.

Low Entropy: Identify a head with low entropy. Does this head seem to focus narrowly on specific syntactic roles (like the next word or previous word)?

High Entropy: Identify a head with high entropy. Does this head seem to "attend" broadly to the whole sentence?

Why is it beneficial for a Transformer to have multiple heads behaving differently, rather than just one "smart" head?

The "Black Box" Revealed: In previous weeks, we treated embeddings as static numbers (Word2Vec). Based on your observations here, explain why a Transformer's understanding of language is considered dynamic compared to those older models.








